# 5. Live Orbit Tracker

Shows each satellite's **current real-time position** and predicts
upcoming overpasses using actual orbit propagation \u2014 independent of
Earth Engine and independent of the historical-search-based forecast in
`4_Future_Overpass_Forecast.ipynb`. Good as a cross-check on that
notebook's predictions, and especially useful for Sentinel-1 where there
may be too little search history to detect a real pattern.

**How it works**: pulls live orbital elements (TLEs) from CelesTrak (a
free public catalog of current satellite orbits) and propagates them with
`skyfield`, the same orbit-mechanics technique used by sites like N2YO.
This needs no Earth Engine project and no login.

**Important caveats**:
- This predicts when each satellite's **swath geometrically crosses your
  AOI** \u2014 a necessary condition for an image, but for **Sentinel-1**
  specifically, whether ESA actually tasks an acquisition on that pass is
  a separate planning decision this can't see. Sentinel-2 and Landsat
  acquire systematically on every qualifying pass, so those predictions
  are more directly reliable.
- "Current position" reflects the moment you run the cell \u2014 re-run it
  to refresh; it doesn't update live on its own.
- Swath widths used for the proximity search are approximate nominal
  figures (Sentinel-1 IW \u2248250 km, Sentinel-2 \u2248290 km, Landsat \u2248185 km).

**Requires**: run `1_AOI_Selection.ipynb` first (only needs the exported
AOI GeoJSON \u2014 no Earth Engine session needed here).

## Load your AOI

No Earth Engine session needed for this notebook \u2014 reads the GeoJSON
directly.

In [1]:
import json
from pathlib import Path

AOI_NAME = "my_aoi"
OUTPUT_DIR = "output"

geojson_path = Path(OUTPUT_DIR) / f"{AOI_NAME}.geojson"
if not geojson_path.exists():
    raise FileNotFoundError(f"{geojson_path} not found \u2014 run 1_AOI_Selection.ipynb first.")

with open(geojson_path) as f:
    aoi_geojson = json.load(f)

aoi_ring = aoi_geojson["features"][0]["geometry"]["coordinates"][0]
AOI_LON = sum(c[0] for c in aoi_ring) / len(aoi_ring)
AOI_LAT = sum(c[1] for c in aoi_ring) / len(aoi_ring)
print(f"AOI centroid: lon={AOI_LON:.4f}, lat={AOI_LAT:.4f}")

AOI centroid: lon=-88.7786, lat=33.4704


## Satellites to track and their approximate swath half-widths

In [2]:
SWATH_HALF_WIDTH_KM = {
    "SENTINEL-1A": 125, "SENTINEL-1C": 125,
    "SENTINEL-2A": 145, "SENTINEL-2B": 145, "SENTINEL-2C": 145,
    "LANDSAT 8": 92, "LANDSAT 9": 92,
}

PLATFORM_COLORS = {
    "SENTINEL-1A": "#14b8a6", "SENTINEL-1C": "#0f766e",
    "SENTINEL-2A": "#eab308", "SENTINEL-2B": "#facc15", "SENTINEL-2C": "#fde047",
    "LANDSAT 8": "#f97316", "LANDSAT 9": "#c2410c",
}

## Fetch live orbital elements (TLEs) from CelesTrak

In [3]:
import requests
from skyfield.api import EarthSatellite, load

ts = load.timescale()
satellites = {}

for name in SWATH_HALF_WIDTH_KM:
    resp = requests.get(
        "https://celestrak.org/NORAD/elements/gp.php",
        params={"NAME": name, "FORMAT": "TLE"},
        timeout=30,
    )
    lines = [l for l in resp.text.strip().split("\n") if l.strip()]
    if len(lines) < 3:
        print(f"{name}: no TLE found, skipping")
        continue
    satellites[name] = EarthSatellite(lines[1], lines[2], lines[0], ts)
    print(f"{name}: TLE loaded")

print(f"\n{len(satellites)} of {len(SWATH_HALF_WIDTH_KM)} satellites loaded.")

SENTINEL-1A: TLE loaded


SENTINEL-1C: TLE loaded


SENTINEL-2A: TLE loaded


SENTINEL-2B: TLE loaded


SENTINEL-2C: TLE loaded


LANDSAT 8: TLE loaded


LANDSAT 9: TLE loaded

7 of 7 satellites loaded.


## Current position snapshot

In [4]:
import math

import pandas as pd
from skyfield.api import wgs84


def _haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlambda / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


t_now = ts.now()
rows = []
for name, sat in satellites.items():
    subpoint = wgs84.subpoint(sat.at(t_now))
    lat, lon = subpoint.latitude.degrees, subpoint.longitude.degrees
    rows.append(
        {
            "satellite": name,
            "lat": lat,
            "lon": lon,
            "altitude_km": subpoint.elevation.km,
            "distance_from_aoi_km": _haversine_km(lat, lon, AOI_LAT, AOI_LON),
        }
    )

snapshot_df = pd.DataFrame(rows).sort_values("distance_from_aoi_km").reset_index(drop=True)
print(f"Snapshot at {t_now.utc_iso()}")
snapshot_df

Snapshot at 2026-08-21T15:58:39Z


,satellite,lat,lon,altitude_km,distance_from_aoi_km
0,LANDSAT 9,38.728046,-79.996889,707.175316,981.191821
1,SENTINEL-1A,63.358259,-132.952668,704.069903,4495.300244
2,SENTINEL-2C,-19.231276,-85.139313,796.472091,5872.869880
3,SENTINEL-1C,-67.678443,-169.991468,722.166292,13065.638340
4,SENTINEL-2B,19.037463,94.888639,791.103831,14163.523711
5,SENTINEL-2A,-17.860801,100.690445,796.138767,18039.363808
6,LANDSAT 8,-38.796041,99.993459,717.932194,19030.351558


## Live map \u2014 current positions and near-term ground tracks

Each satellite's track spans roughly 20 minutes before to 100 minutes
after now, so you can see which direction it's heading. Toggle
satellites on/off with the layer control (top right).

In [5]:
import folium

track_map = folium.Map(location=[AOI_LAT, AOI_LON], zoom_start=3, tiles="CartoDB positron")

folium.Polygon(
    locations=[(c[1], c[0]) for c in aoi_ring],
    color="#ef4444",
    weight=3,
    fill=True,
    fill_opacity=0.15,
    tooltip="AOI",
).add_to(track_map)

track_offsets_min = [m / 60 for m in range(-20 * 60, 100 * 60, 30)]  # every 30s, -20..+100 min
track_times = ts.tt_jd(t_now.tt + [m / 1440 for m in track_offsets_min])

for name, sat in satellites.items():
    color = PLATFORM_COLORS.get(name, "#3388ff")
    group = folium.FeatureGroup(name=name)

    subpoints = wgs84.subpoint(sat.at(track_times))
    lats, lons = subpoints.latitude.degrees, subpoints.longitude.degrees
    # split into separate segments wherever the track crosses the antimeridian
    segment = [(lats[0], lons[0])]
    for i in range(1, len(lats)):
        if abs(lons[i] - lons[i - 1]) > 180:
            folium.PolyLine(segment, color=color, weight=2, opacity=0.7).add_to(group)
            segment = []
        segment.append((lats[i], lons[i]))
    folium.PolyLine(segment, color=color, weight=2, opacity=0.7).add_to(group)

    now_row = snapshot_df[snapshot_df["satellite"] == name].iloc[0]
    folium.CircleMarker(
        location=(now_row["lat"], now_row["lon"]),
        radius=6,
        color=color,
        fill=True,
        fill_opacity=1,
        tooltip=f"{name} \u2014 now, {now_row['distance_from_aoi_km']:.0f} km from AOI",
    ).add_to(group)

    group.add_to(track_map)

folium.LayerControl().add_to(track_map)
track_map

## Physics-based overpass forecast

Propagates each satellite's orbit forward and finds every time its ground
track passes within swath range of the AOI \u2014 a prediction derived purely
from orbital mechanics, not from search history.

In [6]:
import numpy as np

FORECAST_DAYS = 30
STEP_SECONDS = 30

n_samples = int(FORECAST_DAYS * 86400 / STEP_SECONDS)
offsets_days = np.arange(n_samples) * STEP_SECONDS / 86400.0
forecast_times = ts.tt_jd(t_now.tt + offsets_days)


def _haversine_km_vec(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dlambda / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))


orbital_forecast_rows = []
for name, sat in satellites.items():
    half_width = SWATH_HALF_WIDTH_KM[name]
    subpoints = wgs84.subpoint(sat.at(forecast_times))
    lats, lons = subpoints.latitude.degrees, subpoints.longitude.degrees
    dist = _haversine_km_vec(lats, lons, AOI_LAT, AOI_LON)

    is_min = (dist[1:-1] < dist[:-2]) & (dist[1:-1] < dist[2:]) & (dist[1:-1] < half_width)
    idx = np.where(is_min)[0] + 1

    for i in idx:
        predicted_dt = forecast_times[i].utc_datetime()
        orbital_forecast_rows.append(
            {
                "satellite": name,
                "predicted_datetime_utc": predicted_dt.replace(tzinfo=None),
                "days_until": offsets_days[i],
                "closest_approach_km": dist[i],
                "swath_half_width_km": half_width,
            }
        )

orbital_forecast_df = pd.DataFrame(orbital_forecast_rows).sort_values("predicted_datetime_utc").reset_index(drop=True)
print(f"{len(orbital_forecast_df)} predicted overpass(es) through {(t_now.utc_datetime() + pd.Timedelta(days=FORECAST_DAYS)).date()}.")
orbital_forecast_df

52 predicted overpass(es) through 2026-09-20.


,satellite,predicted_datetime_utc,days_until,closest_approach_km,swath_half_width_km
0,SENTINEL-2A,2026-08-21 16:44:39.132683,0.031944,88.901856,145
1,SENTINEL-2A,2026-08-22 03:58:09.132688,0.499653,111.570414,145
2,LANDSAT 9,2026-08-23 03:44:39.132696,1.490278,26.820880,92
3,SENTINEL-2C,2026-08-23 04:07:39.132694,1.506250,123.023030,145
4,SENTINEL-1A,2026-08-23 23:32:09.132674,2.314931,62.480892,125
5,LANDSAT 9,2026-08-24 16:31:09.132691,3.022569,83.919527,92
6,SENTINEL-2B,2026-08-24 16:44:09.132685,3.031597,95.500952,145
7,SENTINEL-1C,2026-08-24 23:38:09.132693,3.319097,121.667622,125
8,SENTINEL-2B,2026-08-25 03:57:39.132690,3.499306,115.808272,145
9,SENTINEL-2A,2026-08-25 04:08:09.132693,3.506597,134.894245,145


## Export

In [7]:
out_csv = Path(OUTPUT_DIR) / f"{AOI_NAME}_orbital_overpass_forecast.csv"
orbital_forecast_df.to_csv(out_csv, index=False)
print(f"Saved {len(orbital_forecast_df)} predicted overpasses -> {out_csv.resolve()}")

Saved 52 predicted overpasses -> C:\Users\Say70\OneDrive - Mississippi State University\Desktop\Satellite Data Search\output\my_aoi_orbital_overpass_forecast.csv
